# THEIA — causal host context and one deployed LightGBM
Train-only vocabulary; source context before sampling; bounded two-hop ancestry and host activity.
Offline candidates: alpha {0.4,0.8} × hard-negative weight {1,5}, selected by validation AP then F1.
Only the selected detector is exported to deployment/lightgbm.txt. Scratch remains an offline transfer control.
Validation-only FPR thresholds; inherited labels/caps retained. Test-informed development remains exploratory.


In [ ]:
# Select the versioned THEIA package, never an arbitrary old-domain manifest.
from pathlib import Path
import hashlib, json, os, tarfile
RUN6_MANIFEST = None
if not os.environ.get('OPTC_MANIFEST'):
    root = Path('/kaggle/input')
    manifests = [p for p in root.rglob('manifest.json')
                 if (p.parent / 'pretrained_optc_model/composition.json').exists()] if root.exists() else []
    if len(manifests) > 1:
        raise RuntimeError('Multiple THEIA packages: set OPTC_MANIFEST explicitly')
    if manifests:
        RUN6_MANIFEST = manifests[0]
    elif root.exists():
        archives = list(root.rglob('input.tar'))
        if len(archives) != 1:
            raise RuntimeError('Attach exactly one input.tar package')
        archive_path = archives[0]
        archive_hash = hashlib.sha256(archive_path.read_bytes()).hexdigest()
        target = Path('/kaggle/working') / ('theia_input_' + archive_hash[:16])
        if not (target / '.extracted').exists():
            target.mkdir(parents=True, exist_ok=True)
            with tarfile.open(archive_path) as archive:
                for member in archive:
                    dest = (target / member.name).resolve()
                    if target.resolve() not in dest.parents or not member.isfile():
                        raise ValueError('Unexpected package member')
                    dest.parent.mkdir(parents=True, exist_ok=True)
                    with archive.extractfile(member) as src, dest.open('wb') as out:
                        import shutil
                        shutil.copyfileobj(src, out)
            (target / '.extracted').write_text(archive_hash)
        RUN6_MANIFEST = target / 'manifest.json'


In [ ]:
# EDIT THESE SETTINGS. Keep the runtime section below generated.
from pathlib import Path
import os, sys, json, subprocess
# Check versions before importing binary scientific packages.
from importlib.metadata import version, PackageNotFoundError
from packaging.version import Version
PACKAGE_REPAIR = '%pip install numpy==1.26.4 scipy==1.15.3 scikit-learn==1.6.1 joblib==1.5.1 threadpoolctl==3.6.0 lightgbm==4.6.0 xgboost==3.0.5'
_versions = {}
for package, minimum, maximum in [('numpy','1.23.5','3.0'), ('scipy','1.15','3.0'),
                                  ('scikit-learn','1.3','3.0')]:
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = '0'
    _versions[package] = installed
    if not Version(minimum) <= Version(installed) < Version(maximum):
        raise RuntimeError(f'Unsupported notebook environment: {package}={installed}; expected >= {minimum}, < {maximum}. '
                           'Run this in a separate cell with Internet enabled, then restart the kernel/session and Run All:\n' + PACKAGE_REPAIR)
try:
    lgb_ver = version('lightgbm')
    _versions['lightgbm'] = lgb_ver
    if not Version('3.0') <= Version(lgb_ver) < Version('5.0'):
        raise RuntimeError(f'Unsupported notebook environment: lightgbm={lgb_ver}; expected >= 3.0, < 5.0. '
                           'Run this in a separate cell with Internet enabled, then restart the kernel/session and Run All:\n' + PACKAGE_REPAIR)
except PackageNotFoundError:
    pass
print('Python:', sys.version.split()[0], 'Packages:', _versions, flush=True)

MODEL_NAME = 'lgbm'  # The only deployed Host ML model family.
PROFILE = 'kaggle'
MANIFEST = Path(os.environ['OPTC_MANIFEST']) if os.environ.get('OPTC_MANIFEST') else globals().get('RUN6_MANIFEST')
OUTPUT_ROOT = Path('/kaggle/working') if Path('/kaggle/input').exists() else Path.cwd() / 'output/host_ml_training'
OUTPUT_ROOT = Path(os.environ.get('OPTC_OUTPUT', str(OUTPUT_ROOT)))
TRAIN_TARGET_ROWS = None
NEG_PER_POS = 30
SEED = 1
CPU_THREADS = None  # None = CPUs actually granted by affinity/cgroup quota
DEVICE = os.environ.get('OPTC_DEVICE', 'auto')  # xgb only: auto/cpu/cuda
MAX_MATRIX_GIB = 4
MAX_SAMPLE_NNZ = 1_800_000_000  # safety margin below RF's int32 nnz limit
BOOST_ROUNDS = 300
RF_TREES = 500
BASE_MARGIN_ALPHA = float(os.environ.get('ALPHA_OVERRIDE', os.environ.get('OPTC_ALPHA', '0.8')))
if not 0 <= BASE_MARGIN_ALPHA <= 1:
    raise ValueError('Task-2 alpha must be in [0,1]')
USE_REPLAY = os.environ.get('THEIA_USE_REPLAY', '1') == '1'
HARD_NEGATIVE_WEIGHT = float(os.environ.get('THEIA_HARD_NEGATIVE_WEIGHT', '5'))
if HARD_NEGATIVE_WEIGHT < 1:
    raise ValueError('Hard-negative weight must be >= 1')
# alpha=0 with replay is a replay ablation, not the THEIA-only scratch baseline.
BALANCE_CLASSES = os.environ.get('THEIA_BALANCE_CLASSES', '1') == '1'
DEDUPE_PREDICTION = True  # exact binary rows, unchanged event weights/coverage
DEDUPE_BUDGET_MIB = 128 if PROFILE == 'ec2-100m' else 64
RANKING_BINS = 100_000
SAMPLE_DIR = Path(os.environ.get('OPTC_SAMPLE_DIR', str(OUTPUT_ROOT / ('train-100m-seed01' if TRAIN_TARGET_ROWS else 'train-neg30-seed01'))))

if MANIFEST is None:
    MANIFEST = Path.cwd() / 'IL/dataset_2-THEIA/code/ML/preprocess/output/manifest.json'
MANIFEST = Path(MANIFEST).resolve()
if not (MANIFEST.parent / 'pretrained_optc_model/composition.json').exists():
    raise RuntimeError('Run 6 requires the complete composite checkpoint from input.tar')
OPTC_TEST_MANIFEST = MANIFEST.parent / 'optc_test/manifest.json'
ATLASV2_TEST_MANIFEST = MANIFEST.parent / 'atlasv2_test/manifest.json'
if not all(p.exists() for p in (MANIFEST, OPTC_TEST_MANIFEST, ATLASV2_TEST_MANIFEST)):
    raise RuntimeError('Incomplete THEIA package')

# Require the new experiment, even when OPTC_MANIFEST is explicitly supplied.
_input_meta = json.loads(MANIFEST.read_text())
print(f"Loaded Manifest: {MANIFEST} | Vocab size: {_input_meta.get('vocab_size')}")
_experiment = 'theia-incremental-' + _input_meta['vocabulary_sha256'][:12]
OUTPUT_ROOT = OUTPUT_ROOT / _experiment
SAMPLE_DIR = OUTPUT_ROOT / 'train-neg30-sampling-seed01'

sys.path.insert(0, str(MANIFEST.parent))
# BEGIN EMBEDDED RUNTIME (regenerate with refresh_host_notebooks.py)
RUNTIME_SHA256 = '8ade6922091ce6e7793e00a7d10d470e9a5a7c98613d52a3d8830041fadacfa6'
"""Shared notebook runtime. Embedded by refresh_host_notebooks.py for standalone use.

Only downstream TRAIN samples and model/evaluation state are written. The
published full-data readers, source shards, vocabulary and split stay unchanged.
"""
from pathlib import Path
import contextlib
import hashlib
import inspect
import json
import os
import tempfile
import time
from concurrent.futures import ThreadPoolExecutor
from collections import deque

import numpy as np
import scipy
from scipy import sparse
from scipy.stats import hypergeom
import host_shards as _host_shards
from host_shards import HostShardDataset, decode_shard


def sha256(path):
    """Streaming SHA256 compatible with Python 3.10 and existing packages."""
    result = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            result.update(block)
    return result.hexdigest()


if not hasattr(hashlib, 'file_digest'):
    def _file_digest(fileobj, digest, /, **kwargs):
        h = hashlib.new(digest, **kwargs) if isinstance(digest, str) else digest(**kwargs)
        for block in iter(lambda: fileobj.read(1024 * 1024), b''):
            h.update(block)
        return h
    hashlib.file_digest = _file_digest
    # The uploaded reader also hashes assets and every decoded shard internally.
    # Patch its module binding in memory, never the read-only input file.
    _host_shards.sha256 = sha256

if 'RUNTIME_SHA256' not in globals():
    RUNTIME_SHA256 = hashlib.sha256(Path(__file__).read_bytes()).hexdigest()
READER_SHA256 = sha256(inspect.getfile(decode_shard))

# Optional representation ablation. Only new THEIA directory-relation columns
# are masked; the frozen Task-1 prefix retains its exact original meanings.
DROP_ACTOR_DIRECTORY = os.environ.get('THEIA_DROP_ACTOR_DIRECTORY', '0') == '1'
_vocab = json.loads((MANIFEST.parent / 'assets/vocab_extended.json').read_text())
_base_width = len(json.loads((MANIFEST.parent / 'assets/vocab.json').read_text()))
MASKED_COLUMNS = sorted(i for token, i in _vocab.items() if i >= _base_width and
    token.startswith(('actor_dir::', 'relation::actor_dir->', 'relation::parent_dir->actor_dir='))) if DROP_ACTOR_DIRECTORY else []
_original_decode_shard = decode_shard

def decode_shard(path, width, part):
    X, y, lines = _original_decode_shard(path, width, part)
    if MASKED_COLUMNS:
        X.data[np.isin(X.indices, MASKED_COLUMNS)] = 0
        X.eliminate_zeros()
    return X, y, lines



def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(',', ':')).encode()).hexdigest()


def output_path(path, protected=()):
    path = Path(path).resolve()
    # Portable guard also works when the code lives in an isolated worktree.
    if any(name in path.parts for name in ('host-begnin', 'host-malicious', 'bro', 'ecar-bro')):
        raise ValueError(f'output is inside a read-only dataset: {path}')
    for root in protected:
        if path.is_relative_to(Path(root).resolve()):
            raise ValueError(f'output overlaps input: {path}')
    return path


def atomic_write(path, writer, protected=()):
    path = output_path(path, protected)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, name = tempfile.mkstemp(prefix='.' + path.stem + '-', suffix=path.suffix, dir=path.parent)
    os.close(fd)
    temporary = output_path(name, protected)
    try:
        writer(temporary)
        with temporary.open('rb') as handle:
            os.fsync(handle.fileno())
        os.replace(temporary, path)
        fd = os.open(path.parent, os.O_RDONLY)
        try:
            os.fsync(fd)
        finally:
            os.close(fd)
    finally:
        temporary.unlink(missing_ok=True)


def atomic_json(path, value, protected=()):
    atomic_write(path, lambda p: p.write_text(json.dumps(value, indent=2)), protected)


def cpu_budget(override=None):
    from joblib import cpu_count
    available = max(1, cpu_count())  # cgroup quotas and process affinity, not host CPU count
    return available if override is None else max(1, min(available, int(override)))


def configure_threads(threads):
    for key in ('OMP_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
        os.environ[key] = str(threads)
    for key in ('OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'BLIS_NUM_THREADS'):
        os.environ[key] = '1'
    from threadpoolctl import threadpool_limits
    return threadpool_limits(limits=1, user_api='blas')


def available_bytes():
    try:
        import psutil
        available = psutil.virtual_memory().available
    except ImportError:
        fields = dict(line.split(':', 1) for line in Path('/proc/meminfo').read_text().splitlines())
        available = int(fields['MemAvailable'].split()[0])*1024
    for limit_name, used_name in (
        ('/sys/fs/cgroup/memory.max', '/sys/fs/cgroup/memory.current'),
        ('/sys/fs/cgroup/memory/memory.limit_in_bytes', '/sys/fs/cgroup/memory/memory.usage_in_bytes'),
    ):
        try:
            available = min(available, max(0, int(Path(limit_name).read_text()) - int(Path(used_name).read_text())))
        except (OSError, ValueError):
            pass
    return available


class TrainingDataset(HostShardDataset):
    """Cache resolved parent directories without changing published reader code."""
    def __init__(self, *args, **kwargs):
        self._parents = {}
        super().__init__(*args, **kwargs)

    def path_for(self, part):
        if part['volume'] not in self.roots:
            raise ValueError('unknown volume')
        relative = Path(part['path'])
        if relative.is_absolute() or '..' in relative.parts:
            raise ValueError('shard path escapes volume')
        root = self.roots[part['volume']]
        key = (part['volume'], str(relative.parent))
        if key not in self._parents:
            parent = (root / relative.parent).resolve()
            if not parent.is_relative_to(root):
                raise ValueError('shard parent escapes volume')
            self._parents[key] = parent
        path = self._parents[key] / relative.name
        if path.is_symlink():
            raise ValueError('symlink shard is not supported by the training runtime')
        return path

    @property
    def protected(self):
        return [self.manifest_path.parent, *self.roots.values()]

    def identity(self):
        # Content identities travel with the input package. File facts additionally
        # invalidate interrupted runs when files are replaced without a new manifest.
        facts = []
        for part in self.meta['shards']:
            stat = self.path_for(part).stat()
            if stat.st_size != part['bytes']:
                raise ValueError('source shard size changed')
            facts.append((part['volume'], part['path'], stat.st_size, stat.st_mtime_ns))
        return dict(manifest_sha256=sha256(self.manifest_path), source_facts_sha256=digest(facts),
                    reader_sha256=READER_SHA256, masked_columns=MASKED_COLUMNS,
                    split_sha256=self.meta['split_sha256'], vocabulary_sha256=self.meta['vocabulary_sha256'])


def ordered_prefetch(items, function, workers=1):
    """Decode at most 2*workers ahead. Preserve order for deterministic sampling."""
    items = iter(items)
    with ThreadPoolExecutor(max_workers=max(1, workers)) as pool:
        pending = deque()
        for _ in range(max(1, workers) * 2):
            item = next(items, None)
            if item is not None:
                pending.append((item, pool.submit(function, item)))
        while pending:
            item, future = pending.popleft()
            result = future.result()
            following = next(items, None)
            if following is not None:
                pending.append((following, pool.submit(function, following)))
            yield item, result


def sample_plan(data, target_rows=None, neg_per_pos=30, seed=1):
    info = data.meta['splits']['train']
    if info['label_1'] < 1 or info['label_0'] < 1:
        raise ValueError('TRAIN requires both classes')
    if target_rows is not None and not info['label_1'] < target_rows <= info['rows']:
        raise ValueError('target_rows must exceed positive count and not exceed TRAIN rows')
    if neg_per_pos <= 0:
        raise ValueError('neg_per_pos must be positive')
    rng = np.random.default_rng(seed)
    remaining = info['label_0']
    needed = None if target_rows is None else target_rows - info['label_1']
    keep_p = min(1.0, neg_per_pos * info['label_1'] / info['label_0'])
    entries = []
    for index, part in enumerate(data.meta['shards']):
        if part['split'] != 'train':
            continue
        n = part['label_0']
        if target_rows is None:
            # Exactly the prior Nguyen RNG stream: one draw per negative in order.
            state = rng.bit_generator.state
            k = int(np.count_nonzero(rng.random(n) < keep_p))
        else:
            # numpy.hypergeometric rejects populations above 1e9. SciPy's inverse
            # CDF handles this corpus size without an O(population) allocation.
            if needed == 0 or n == 0:
                k = 0
            elif remaining == n:
                k = needed
            elif needed == remaining:
                k = n
            else:
                u = max(float(rng.random()), np.nextafter(0.0, 1.0))
                k = int(hypergeom.ppf(u, remaining, needed, n))
            state = None
            needed -= k
            remaining -= n
        entries.append(dict(index=index, negatives=k, rng_state=state))
    rows = sum(e['negatives'] + data.meta['shards'][e['index']]['label_1'] for e in entries)
    if target_rows is not None and (needed != 0 or rows != target_rows):
        raise ValueError('exact sample plan count mismatch')
    return dict(method='exact_stratified_train_sample' if target_rows is not None else 'random_negative_undersampling',
                seed=seed, target_rows=target_rows, neg_per_pos=neg_per_pos, keep_probability=keep_p,
                rows=rows, positives=info['label_1'], negatives=rows-info['label_1'], entries=entries)


def selected_rows(y, entry, plan):
    neg = np.flatnonzero(y == 0)
    if plan['target_rows'] is None:
        rng = np.random.default_rng()
        rng.bit_generator.state = entry['rng_state']
        kept = neg[rng.random(len(neg)) < plan['keep_probability']]
    else:
        rng = np.random.default_rng(np.random.SeedSequence([plan['seed'], entry['index']]))
        kept = neg[rng.choice(len(neg), size=entry['negatives'], replace=False)]
    if len(kept) != entry['negatives']:
        raise ValueError('sample selection differs from plan')
    return np.sort(np.concatenate((np.flatnonzero(y == 1), kept))).astype(np.int32)


def prepare_sample(data, directory, target_rows=None, neg_per_pos=30, seed=1,
                   workers=1, max_nnz=1_800_000_000, checkpoint_every=64):
    directory = output_path(directory, data.protected)
    identity = data.identity()
    config = dict(**identity, runtime_sha256=RUNTIME_SHA256, numpy=np.__version__, scipy=scipy.__version__,
                  target_rows=target_rows, neg_per_pos=neg_per_pos, seed=seed, max_nnz=max_nnz)
    directory.mkdir(parents=True, exist_ok=True)
    # Concurrent models should share a completed sample, not mutate it concurrently.
    import fcntl
    with output_path(directory/'sample.lock', data.protected).open('a+b') as lock:
        fcntl.flock(lock, fcntl.LOCK_EX)
        plan_path = directory/'plan.json'
        if plan_path.exists():
            saved = json.loads(plan_path.read_text())
            if saved['config'] != config:
                raise ValueError('sample configuration/source/runtime changed; use a new sample directory')
            plan = saved['plan']
        else:
            print('Planning shared TRAIN sample...', flush=True)
            plan = sample_plan(data, target_rows, neg_per_pos, seed)
            atomic_json(plan_path, dict(config=config, plan=plan), data.protected)
        # File timestamps validate local resume but do not change which events
        # were selected when an identical package is relocated to another VM.
        sample_id = digest(dict(config={k:v for k,v in config.items() if k != 'source_facts_sha256'}, plan=plan))
        state_path = directory/'manifest.json'
        state = json.loads(state_path.read_text()) if state_path.exists() else dict(
            sample_id=sample_id, complete=False, config=config, rows=0, nnz=0, positives=0, parts=[])
        if state['sample_id'] != sample_id:
            raise ValueError('sample checkpoint identity mismatch')
        for part in state['parts']:
            path = output_path(directory/part['path'], data.protected)
            if not path.is_relative_to(directory) or sha256(path) != part['sha256']:
                raise ValueError('sample shard checksum mismatch')
        completed = {p['index'] for p in state['parts']}
        todo = [e for e in plan['entries'] if e['index'] not in completed
                and e['negatives'] + data.meta['shards'][e['index']]['label_1'] > 0]

        def load(entry):
            source = data.meta['shards'][entry['index']]
            X, y, _ = decode_shard(data.path_for(source), data.width, source)
            rows = selected_rows(y, entry, plan)
            return X[rows], y[rows], rows

        started = time.monotonic()
        for i, (entry, (X, y, rows)) in enumerate(ordered_prefetch(todo, load, workers), 1):
            if state['nnz'] + X.nnz > max_nnz:
                raise MemoryError('sample exceeds nnz safety limit; choose a smaller common target, never truncate rows')
            path = output_path(directory/f'parts/{entry["index"]:06d}.npz', data.protected)
            atomic_write(path, lambda p: np.savez_compressed(p, indices=X.indices.astype(np.int32),
                         indptr=X.indptr.astype(np.int32), label=y, source_rows=rows), data.protected)
            state['parts'].append(dict(index=entry['index'], path=str(path.relative_to(directory)),
                                       rows=len(y), nnz=X.nnz, positives=int(y.sum()), sha256=sha256(path)))
            state['rows'] += len(y)
            state['nnz'] += X.nnz
            state['positives'] += int(y.sum())
            if i % checkpoint_every == 0:
                atomic_json(state_path, state, data.protected)
                print(f'Sampling {i}/{len(todo)} remaining shards; kept {state["rows"]:,} rows; '
                      f'{time.monotonic()-started:.1f}s', flush=True)
        if state['rows'] != plan['rows'] or state['positives'] != plan['positives']:
            raise ValueError('sample totals differ from the plan')
        state.update(complete=True, method=plan['method'], seed=seed, width=data.width)
        atomic_json(state_path, state, data.protected)
        print(f'Sample ready: {state["rows"]:,} rows, {state["positives"]:,} positives, '
              f'{state["nnz"]:,} nnz; identity={sample_id}', flush=True)
        return state


def materialize_sample(directory, state, max_bytes, memory_fraction=0.55):
    if not state['complete']:
        raise ValueError('sample is incomplete')
    n, nz = state['rows'], state['nnz']
    if max(n, nz, state['width']) >= 2**31:
        raise MemoryError('sample does not fit RF-compatible int32 CSR')
    required = 8*nz + 4*(n+1) + n
    if required > min(max_bytes, available_bytes()*memory_fraction):
        raise MemoryError(f'Sampled CSR needs {required/2**30:.2f} GiB before model/temporary memory')
    values = np.ones(nz, dtype=np.float32)
    indices = np.empty(nz, dtype=np.int32)
    indptr = np.empty(n+1, dtype=np.int32)
    labels = np.empty(n, dtype=np.uint8)
    indptr[0] = 0
    r = z = 0
    for part in state['parts']:
        path = Path(directory)/part['path']
        if sha256(path) != part['sha256']:
            raise ValueError('sample shard checksum mismatch')
        with np.load(path, allow_pickle=False) as chunk:
            nr, nnz = part['rows'], part['nnz']
            indices[z:z+nnz] = chunk['indices']
            indptr[r+1:r+nr+1] = chunk['indptr'][1:] + z
            labels[r:r+nr] = chunk['label']
        r += nr
        z += nnz
    if r != n or z != nz:
        raise ValueError('sample materialization count mismatch')
    return sparse.csr_matrix((values, indices, indptr), shape=(n,state['width']), copy=False), labels


def unique_binary_rows(X, budget_bytes=64*2**20):
    """Exact in-batch deduplication, no hash collisions or changed row weights.

    Group by row length, then compare exact column-index bytes. Fall back to
    unchanged input if temporary buffers exceed budget. Never densify features.
    """
    n = X.shape[0]
    if not n:
        return X, np.empty(0, dtype=np.int64)
    counts = np.diff(X.indptr)
    if counts.max(initial=0) == 0:
        return X[:1], np.zeros(n, dtype=np.int64)
    required = X.nnz*24 + n*64
    if required > budget_bytes:
        return X, None
    if not X.has_canonical_format or not np.all(X.data == 1):
        raise ValueError('deduplication requires canonical binary presence CSR')
    order = np.argsort(counts, kind='stable')
    lengths, starts, sizes = np.unique(counts[order], return_index=True, return_counts=True)
    inverse = np.empty(n, dtype=np.int64)
    first_parts = []
    offset = 0
    for width, start, size in zip(lengths, starts, sizes):
        rows = order[start:start+size]
        if width == 0:
            first, local_inverse = rows[:1], np.zeros(size, dtype=np.int64)
        elif size == 1:
            first, local_inverse = rows, np.zeros(1, dtype=np.int64)
        else:
            positions = X.indptr[rows].astype(np.int64)[:,None] + np.arange(width)
            keys = np.ascontiguousarray(X.indices[positions], dtype=np.int32)
            records = keys.view(np.dtype((np.void, 4*int(width)))).ravel()
            _, local_first, local_inverse = np.unique(records, return_index=True, return_inverse=True)
            first = rows[local_first]
        first_parts.append(first)
        inverse[rows] = local_inverse + offset
        offset += len(first)
    first = np.concatenate(first_parts)
    if len(first) == n:
        return X, None
    return X[first], inverse


class ExactPredictor:
    def __init__(self, predict, enabled=True, budget_bytes=64*2**20):
        self.predict = predict
        self.enabled = enabled
        self.budget_bytes = budget_bytes
        self.rows = self.scored = 0

    def __call__(self, X):
        unique, inverse = unique_binary_rows(X, self.budget_bytes) if self.enabled else (X, None)
        p = np.asarray(self.predict(unique)).reshape(-1)
        if len(p) != unique.shape[0]:
            raise ValueError('prediction length mismatch')
        self.rows += X.shape[0]
        self.scored += unique.shape[0]
        return p if inverse is None else p[inverse]


"""Reconstruct the Task-1 predictor; a residual booster is not a standalone model."""
import hashlib
import json
from pathlib import Path
import numpy as np
from scipy import sparse
from scipy.special import expit


class FrozenAtlasModel:
    def __init__(self, residual_path):
        import lightgbm as lgb
        residual_path = Path(residual_path)
        config_path = residual_path.parent / 'composition.json'
        config = json.loads(config_path.read_text())
        def verified(entry):
            p = (config_path.parent / entry['path']).resolve()
            if hashlib.sha256(p.read_bytes()).hexdigest() != entry['sha256']:
                raise ValueError(f'Frozen model hash mismatch: {p}')
            return p
        anchor = verified(config['anchor'])
        residual = verified(config['residual'])
        if residual != residual_path.resolve():
            raise ValueError('Residual path differs from composition manifest')
        self.anchor = lgb.Booster(model_file=str(anchor))
        self.residual = lgb.Booster(model_file=str(residual))
        self.alpha = float(config['alpha'])
        self.composition_sha256 = hashlib.sha256(config_path.read_bytes()).hexdigest()

    def num_feature(self):
        return self.residual.num_feature()

    def num_trees(self):
        return self.anchor.num_trees() + self.residual.num_trees()

    @staticmethod
    def align(X, width):
        if X.shape[1] >= width:
            return X[:, :width]
        return sparse.hstack([X, sparse.csr_matrix((X.shape[0], width-X.shape[1]))], format='csr')

    def predict(self, X, raw_score=False, num_threads=1, **kwargs):
        z = self.alpha * self.anchor.predict(self.align(X, self.anchor.num_feature()),
                                             raw_score=True, num_threads=num_threads)
        z += self.residual.predict(self.align(X, self.residual.num_feature()),
                                   raw_score=True, num_threads=num_threads)
        return z if raw_score else expit(z)


THEIA_TARGET_FPR = 0.005  # Fixed before Run 6; empirical sampled-validation budget
THEIA_MAX_VALIDATION_FPR = 0.01  # Upper bound constraint for validation F1 calibration

"""Validation-negative-only and F1-constrained operating points; ties are never split to meet a budget."""
import numpy as np


def select_fpr_threshold(negative_histogram, target_fpr=0.005):
    negative = np.asarray(negative_histogram)
    if negative.ndim != 1 or len(negative) < 2 or np.any(negative < 0):
        raise ValueError('Invalid negative histogram')
    if not 0 <= target_fpr < 1 or not negative.sum():
        raise ValueError('Need validation negatives and FPR in [0,1)')
    fp = negative[::-1].cumsum()[::-1]
    allowed = int(np.floor(target_fpr * int(negative.sum())))
    candidates = np.flatnonzero(fp <= allowed)
    if not len(candidates):
        return float(np.nextafter(1.0, np.inf)), len(negative)
    index = int(candidates[0])
    return index / (len(negative) - 1), index


def select_validation_operating_point(positive_histogram, negative_histogram, max_fpr=0.01, plateau_rel_tol=0.03):
    """Validation F1-optimal operating point under an empirical FPR ceiling with Left-Edge Plateau selection."""
    pos = np.asarray(positive_histogram)
    neg = np.asarray(negative_histogram)
    if pos.ndim != 1 or neg.ndim != 1 or len(pos) != len(neg) or len(pos) < 2:
        raise ValueError('Invalid histograms')
    total_p, total_n = int(pos.sum()), int(neg.sum())
    if not total_p or not total_n:
        raise ValueError('Need both positive and negative validation samples')
    tps, fps = pos[::-1].cumsum()[::-1], neg[::-1].cumsum()[::-1]
    f1s = 2 * tps / np.maximum(2 * tps + fps + (total_p - tps), 1)
    if max_fpr is not None and 0 < max_fpr < 1:
        allowed = int(np.floor(max_fpr * total_n))
        candidates = np.flatnonzero(fps <= allowed)
        if len(candidates):
            peak_f1 = np.max(f1s[candidates])
            if plateau_rel_tol is not None and plateau_rel_tol > 0 and peak_f1 > 0:
                plateau_candidates = candidates[f1s[candidates] >= (1.0 - plateau_rel_tol) * peak_f1]
                best = int(plateau_candidates[0])
            else:
                best = int(candidates[np.argmax(f1s[candidates])])
        else:
            best = int(np.argmax(f1s))
    else:
        if plateau_rel_tol is not None and plateau_rel_tol > 0:
            peak_f1 = np.max(f1s)
            plateau_candidates = np.flatnonzero(f1s >= (1.0 - plateau_rel_tol) * peak_f1)
            best = int(plateau_candidates[0]) if len(plateau_candidates) else int(np.argmax(f1s))
        else:
            best = int(np.argmax(f1s))
    return float(best / (len(pos) - 1)), best


def evaluate_resumable(data, split, predict, state_path, model_identity, threshold=None,
                       bins=100000, workers=1, checkpoint_every=32, diagnostics=None):
    if split not in ('val', 'test') or bins < 2 or (split == 'test' and threshold is None):
        raise ValueError('test requires a frozen validation threshold')
    if threshold is not None and not 0 <= threshold <= np.nextafter(1.0, np.inf):
        raise ValueError('threshold outside [0,1]')
    calibration_source = 'validation_f1_max_constrained' if threshold is None else 'provided_frozen_threshold'
    state_path = output_path(state_path, data.protected)
    config = dict(input=data.identity(), model=model_identity, runtime_sha256=RUNTIME_SHA256,
                  split=split, bins=bins, threshold=threshold, target_fpr=THEIA_MAX_VALIDATION_FPR,
                  diagnostics=diagnostics.config if diagnostics is not None else None)
    signature = digest(config)
    positive = np.zeros(bins+1, dtype=np.int64)
    negative = np.zeros(bins+1, dtype=np.int64)
    counts = np.zeros(4, dtype=np.int64)  # TP, FP, FN, TN at frozen threshold
    next_part = 0
    if state_path.exists():
        with np.load(state_path, allow_pickle=False) as saved:
            if str(saved['signature']) != signature:
                raise ValueError('evaluation checkpoint model/input/config mismatch; use a new run directory')
            positive, negative, counts = saved['positive'], saved['negative'], saved['counts']
            next_part = int(saved['next_part'])
            if diagnostics is not None:
                diagnostics.restore(json.loads(str(saved['diagnostics'])))
    parts = [p for p in data.meta['shards'] if p['split'] == split]
    started = time.monotonic()
    print(f'{split}: resume shard {next_part}/{len(parts)}; full split {data.meta["splits"][split]["rows"]:,} rows', flush=True)

    def save():
        atomic_write(state_path, lambda p: np.savez_compressed(p, signature=signature, positive=positive,
                     negative=negative, counts=counts, next_part=next_part,
                     diagnostics=json.dumps(diagnostics.state if diagnostics is not None else None)), data.protected)

    def load(part):
        return decode_shard(data.path_for(part), data.width, part)

    for part, (X,y,lines) in ordered_prefetch(parts[next_part:], load, workers):
        p = np.asarray(predict(X)).reshape(-1)
        if len(p) != len(y) or not np.all(np.isfinite(p)) or np.any((p < 0) | (p > 1)):
            raise ValueError('invalid probabilities')
        if diagnostics is not None:
            diagnostics.update(part, X, y, lines, p, threshold)
        bucket = np.minimum((p.astype(np.float64)*bins).astype(np.int64), bins)
        positive += np.bincount(bucket[y == 1], minlength=bins+1)
        negative += np.bincount(bucket[y == 0], minlength=bins+1)
        if threshold is not None:
            predicted = p.astype(np.float64) >= threshold
            counts += (np.sum(predicted & (y == 1)), np.sum(predicted & (y == 0)),
                       np.sum(~predicted & (y == 1)), np.sum(~predicted & (y == 0)))
        next_part += 1
        if next_part % checkpoint_every == 0:
            save()
            print(f'{split}: {next_part}/{len(parts)} shards; '
                  f'{int(positive.sum()+negative.sum()):,} rows; {time.monotonic()-started:.1f}s this session', flush=True)
    save()
    total_p, total_n = int(positive.sum()), int(negative.sum())
    expected = data.meta['splits'][split]
    if total_p != expected['label_1'] or total_n != expected['label_0']:
        raise ValueError('evaluation totals differ from full split')
    if not total_p or not total_n:
        raise ValueError('both classes required for evaluation')
    tps, fps = positive[::-1].cumsum()[::-1], negative[::-1].cumsum()[::-1]
    precisions = tps / np.maximum(tps+fps, 1)
    if threshold is None:
        threshold, best = select_validation_operating_point(positive, negative, THEIA_MAX_VALIDATION_FPR)
        counts = ((int(tps[best]), int(fps[best]), total_p-int(tps[best]), total_n-int(fps[best]))
                  if best < len(tps) else (0, 0, total_p, total_n))
    tp, fp, fn, tn = map(int, counts)
    precision, recall = tp/max(tp+fp, 1), tp/total_p
    return dict(split=split, rows=total_p+total_n, threshold=float(threshold), threshold_source=calibration_source,
                target_validation_fpr=THEIA_MAX_VALIDATION_FPR,
                precision=precision, recall=recall, f1=2*tp/max(2*tp+fp+fn, 1), fpr=fp/total_n,
                confusion=dict(tp=tp,fp=fp,fn=fn,tn=tn), ranking_bins=bins,
                pr_auc_binned=float(np.sum(precisions*positive/total_p)),
                roc_auc_binned=float(np.sum(positive.astype(np.float64)*(negative.cumsum()-negative/2))/(total_p*total_n)),
                note='Sampled evaluation cohort; validation-negative FPR policy; test confusion counted directly; binned AUC. FPR budget is empirical, not a population guarantee.')


@contextlib.contextmanager
def phase(name):
    started = time.monotonic()
    print(f'BEGIN {name}', flush=True)
    try:
        yield
    finally:
        print(f'END {name}: {time.monotonic()-started:.1f}s', flush=True)


def fit_sample_model(data, sample_dir, sample, output, model_name, threads,
                     max_matrix_bytes, device='cpu', balance_classes=False,
                     rounds=300, rf_trees=500, checkpoint_every=25, predict_threads=None,
                     init_model_path=None, base_margin_alpha=None, hard_negative_weight=None):
    """Resume boosting rounds / RF warm-start trees; publish model before metrics.

    Two alternating model files keep the last published checkpoint intact while
    saving its successor. A complete model skips matrix loading and fitting.
    """
    base_margin_alpha = BASE_MARGIN_ALPHA if base_margin_alpha is None else base_margin_alpha
    hard_negative_weight = HARD_NEGATIVE_WEIGHT if hard_negative_weight is None else hard_negative_weight
    import importlib
    import pickle
    library_name = {'xgb':'xgboost', 'lgbm':'lightgbm', 'rf':'sklearn'}[model_name]
    library = importlib.import_module(library_name)
    output = output_path(output, data.protected)
    output.mkdir(parents=True, exist_ok=True)
    positive_weight = (sample['rows']-sample['positives'])/sample['positives'] if balance_classes else 1.0
    if model_name == 'lgbm':
        params = dict(objective='binary', metric='None', num_leaves=31, learning_rate=0.03,
                      seed=1, num_threads=threads, verbosity=1, force_col_wise=True,
                      deterministic=True, scale_pos_weight=positive_weight,
                      max_delta_step=2.0, reg_lambda=5.0)
        target = rounds
        suffix = '.txt'
    elif model_name == 'xgb':
        params = dict(objective='binary:logistic', eval_metric='aucpr', tree_method='hist',
                      device=device, max_depth=8, eta=0.05, seed=1, nthread=threads,
                      max_bin=16, scale_pos_weight=positive_weight)
        target = rounds
        suffix = '.ubj'
    else:
        params = dict(max_depth=20, min_samples_leaf=5, max_features='sqrt',
                      class_weight={0:sample['rows']/(2*(sample['rows']-sample['positives'])),
                                    1:sample['rows']/(2*sample['positives'])},
                      random_state=1, warm_start=True)
        target = rf_trees
        suffix = '.pkl'
    init_sha = sha256(Path(init_model_path)) if init_model_path and Path(init_model_path).exists() else None
    identity = dict(sample_id=sample['sample_id'], runtime_sha256=RUNTIME_SHA256,
                    library_version=library.__version__, model=model_name, target=target,
                    init_model_sha=init_sha,
                    composition_sha256=sha256(Path(init_model_path).parent / 'composition.json') if init_model_path else None,
                    base_margin_alpha=base_margin_alpha if init_model_path else None,
                    hard_negative_weight=hard_negative_weight if init_model_path else None,
                    use_replay=USE_REPLAY if init_model_path else False,
                    replay_sha256=sha256(MANIFEST.parent / 'optc_replay_buffer.npz') if init_model_path and USE_REPLAY else None,
                    params={k:v for k,v in params.items() if k not in ('nthread','num_threads')})
    signature = digest(identity)
    latest_path = output/'latest.json'
    latest = json.loads(latest_path.read_text()) if latest_path.exists() else None
    model = None
    done = 0
    generation = 0
    if latest is not None:
        if latest['signature'] != signature:
            raise ValueError('model checkpoint configuration/sample/library changed; use a new model directory')
        path = output_path(output/latest['path'], data.protected)
        if not path.is_relative_to(output) or sha256(path) != latest['sha256']:
            raise ValueError('model checkpoint checksum mismatch')
        if model_name == 'lgbm':
            model = library.Booster(model_file=str(path))
        elif model_name == 'xgb':
            model = library.Booster(model_file=str(path))
        else:
            # Only load this runtime's own, checksummed local checkpoint.
            with path.open('rb') as handle:
                model = pickle.load(handle)
        done, generation = latest['completed'], latest['generation']
    base_booster = None
    if init_model_path is not None and Path(init_model_path).exists():
        init_p = Path(init_model_path).resolve()
        if model_name == 'lgbm':
            base_booster = FrozenAtlasModel(init_p)
            print(f"{model_name}: loaded OpTC base booster from {init_p} ({base_booster.num_trees()} trees) for Base Margin Scaling (alpha={base_margin_alpha})", flush=True)
        elif model_name == 'xgb':
            base_booster = library.Booster(model_file=str(init_p))
            print(f"{model_name}: loaded OpTC base booster from {init_p} ({base_booster.num_boosted_rounds()} rounds) for Base Margin Scaling (alpha={base_margin_alpha})", flush=True)

    def save(current, completed, complete=False):
        nonlocal generation, latest
        generation += 1
        path = output/f'model-slot-{generation % 2}{suffix}'
        if model_name == 'rf':
            def writer(p):
                with p.open('wb') as handle:
                    pickle.dump(current, handle, protocol=5)
        else:
            writer = lambda p: current.save_model(str(p))
        atomic_write(path, writer, data.protected)
        latest = dict(signature=signature, identity=identity, path=path.name, sha256=sha256(path),
                      completed=completed, complete=complete, generation=generation, threads=threads)
        atomic_json(latest_path, latest, data.protected)
        print(f'{model_name}: checkpoint {completed}/{target}, complete={complete}', flush=True)

    if latest and done >= target and not latest['complete']:
        save(model, target, complete=True)
    if not latest or not latest['complete']:
        with phase('load sampled CSR'):
            X, y = materialize_sample(sample_dir, sample, max_matrix_bytes)
        print(f'{model_name}: training {sample["rows"]:,} rows, device={device}, CPU threads={threads}', flush=True)
        with phase(f'{model_name} training'):
            if model_name == 'lgbm':
                if base_booster is not None:
                    # Domain-Incremental Learning with Attack Prototype Re-anchoring Experience Replay
                    replay_path = None
                    for cand_rep in [
                        MANIFEST.parent / 'optc_replay_buffer.npz',
                        Path('/kaggle/input/datasets/ngkimngn/ml-il-theia/optc_replay_buffer.npz'),
                        Path('/kaggle/input/ml-il-theia/optc_replay_buffer.npz'),
                        *(list(Path('/kaggle/input').rglob('optc_replay_buffer.npz')) if Path('/kaggle/input').exists() else []),
                        Path.cwd() / 'IL/dataset_2-THEIA/code/ML/preprocess/output/optc_replay_buffer.npz',
                        Path.cwd() / 'IL/dataset_1-ATLASv2/code/ML/preprocess/output/optc_replay_buffer.npz',
                    ]:
                        if cand_rep.exists():
                            replay_path = cand_rep.resolve()
                            break

                    if replay_path is not None and USE_REPLAY:
                        with np.load(replay_path, allow_pickle=False) as buf:
                            X_rep = sparse.csr_matrix((buf['data'], buf['indices'], buf['indptr']), shape=buf['shape'])
                            y_rep = buf['labels']
                        if X_rep.shape[1] < X.shape[1]:
                            pad_cols = X.shape[1] - X_rep.shape[1]
                            X_rep = sparse.hstack([X_rep, sparse.csr_matrix((X_rep.shape[0], pad_cols), dtype=X_rep.dtype)], format='csr')
                        X_train = sparse.vstack([X, X_rep], format='csr')
                        y_train = np.concatenate([y, y_rep])
                        print(f"Incremental Boosting with Experience Replay: Merged {X_rep.shape[0]:,} OpTC replay prototypes ({int(y_rep.sum()):,} attack, {len(y_rep)-int(y_rep.sum()):,} neg). Total train: {X_train.shape[0]:,} rows.", flush=True)
                    else:
                        X_train = X
                        y_train = y
                        print("OpTC replay buffer not found; proceeding with ATLASv2 training set only.", flush=True)

                    base_n_feats = base_booster.num_feature()
                    raw_optc = base_booster.predict(X_train[:, :base_n_feats], raw_score=True, num_threads=threads, predict_disable_shape_check=True)
                    init_score = base_margin_alpha * raw_optc
                    # Resuming residual boosting requires both previous residual predictions
                    # in gradients and the previous trees in the resulting ensemble.
                    if model is not None:
                        raise RuntimeError('Partial residual-model resume needs a composite base+residual init predictor; restart in a new output directory. Complete checkpoints remain loadable.')
                    sample_weights = np.ones(len(y_train), dtype=np.float32)
                    prob_optc = 1.0 / (1.0 + np.exp(-raw_optc))
                    hard_neg = (y_train == 0) & (prob_optc > 0.5)
                    sample_weights[hard_neg] = hard_negative_weight
                    n_hn = int(np.sum(hard_neg))
                    print(f"Hard-Negative Mining upweighted {n_hn:,} false alarms (y=0, OpTC>0.5) to {hard_negative_weight}x weight", flush=True)
                    train = library.Dataset(X_train, label=y_train, init_score=init_score, weight=sample_weights, free_raw_data=True, params=params)
                    del X_train, y_train
                else:
                    train = library.Dataset(X, label=y, free_raw_data=True, params=params)
                del X, y

                def checkpoint(env):
                    completed = env.iteration+1
                    if completed % checkpoint_every == 0:
                        save(env.model, completed)
                checkpoint.order = 20
                model = library.train(params, train, num_boost_round=max(0,target-done),
                                      init_model=model, callbacks=[library.log_evaluation(period=checkpoint_every), checkpoint])
                del train
            elif model_name == 'xgb':
                train = library.QuantileDMatrix(X, label=y, max_bin=params['max_bin'], nthread=threads)
                del X, y

                class Checkpoint(library.callback.TrainingCallback):
                    def after_iteration(self, booster, epoch, evals_log):
                        completed = done+epoch+1
                        if completed % checkpoint_every == 0:
                            save(booster, completed)
                        return False
                model = library.train(params, train, num_boost_round=max(0,target-done),
                                      xgb_model=model, callbacks=[Checkpoint()])
                del train
            else:
                from sklearn.ensemble import RandomForestClassifier
                X_csc = X.tocsc()
                del X
                # Bootstrap and per-tree buffers grow with sample rows. Avoid
                # launching one full-size tree per CPU when RAM cannot hold them.
                per_tree = sample['rows']*32 + 256*2**20
                fit_jobs = max(1, min(threads, int(available_bytes()*0.5/per_tree)))
                print(f'RF fit workers={fit_jobs}; inference workers={threads}', flush=True)
                if model is None:
                    model = RandomForestClassifier(**params, n_estimators=0, n_jobs=fit_jobs)
                for count in range(done, target, checkpoint_every):
                    completed = min(target, count+checkpoint_every)
                    model.set_params(n_estimators=completed, n_jobs=fit_jobs)
                    model.fit(X_csc, y)
                    save(model, completed)
                del X_csc, y
        save(model, target, complete=True)
    else:
        print(f'{model_name}: using completed model, skipping TRAIN load/fit', flush=True)
    # CPU input is evaluated on CPU explicitly; avoid XGBoost's device-mismatch
    # fallback on every shard. Training can still use one CUDA GPU.
    predict_threads = threads if predict_threads is None else predict_threads
    if model_name == 'xgb':
        model.set_param({'device':'cpu', 'nthread':predict_threads})
        predict = lambda X: model.inplace_predict(X)
    elif model_name == 'lgbm':
        if base_booster is not None:
            base_n_feats = base_booster.num_feature()
            predict = lambda X: 1.0 / (1.0 + np.exp(-(base_margin_alpha * base_booster.predict(X[:, :base_n_feats], raw_score=True, num_threads=predict_threads, predict_disable_shape_check=True) + model.predict(X, raw_score=True, num_threads=predict_threads))))
        else:
            predict = lambda X: model.predict(X, num_threads=predict_threads)
    else:
        model.set_params(n_jobs=predict_threads)
        predict = lambda X: model.predict_proba(X)[:,1]
    return model, predict, latest



class XGBDiagnostics:
    """Bounded stratified uniform probes and extreme-score examples.

    State is published inside the evaluation checkpoint, so metrics and diagnostics
    always resume at the same shard. No extra full validation/test scan is needed.
    """
    def __init__(self, data, probe_negative=10000, probe_positive=1000,
                 examples=100, seed=17, max_probe_nnz=1000000, model_name=None):
        if min(probe_negative, probe_positive, examples, max_probe_nnz) < 1:
            raise ValueError('diagnostic limits must be positive')
        self.data = data
        self.config = dict(probe_negative=probe_negative, probe_positive=probe_positive,
                           examples=examples, seed=seed, max_probe_nnz=max_probe_nnz,
                           model_name=model_name)
        self.rng = np.random.default_rng(seed)
        self.sources = {s['source_id']:s for s in data.meta['sources']}
        self.state = dict(probe=[[], []], extremes=[[], []], rows=[0,0],
                          empty_rows=[0,0], groups={}, rng=self.rng.bit_generator.state)

    def restore(self, state):
        self.state = state
        self.rng.bit_generator.state = state['rng']

    def update(self, part, X, y, lines, scores, threshold=None):
        scores = np.asarray(scores, dtype=np.float64)
        source = self.sources.get(part['source_id'], {})
        source_meta = dict(source_id=part['source_id'], day=source.get('day','unknown'),
                           host=source.get('file_host','unknown'), shard=part.get('path',''))
        sizes = np.diff(X.indptr)
        for label in (0,1):
            rows = np.flatnonzero(y == label)
            self.state['rows'][label] += len(rows)
            self.state['empty_rows'][label] += int(np.count_nonzero(sizes[rows] == 0))
            for name, limit, keys in (
                ('probe', self.config['probe_positive' if label else 'probe_negative'], self.rng.random(len(rows))),
                ('extremes', self.config['examples'], scores[rows] if label else -scores[rows]),
            ):
                previous = self.state[name][label]
                # Keep only candidates that can enter the current bounded pool.
                candidates = np.flatnonzero(keys < previous[-1]['priority']) if len(previous) == limit else np.arange(len(rows))
                if len(candidates) > limit:
                    candidates = candidates[np.argsort(keys[candidates], kind='stable')[:limit]]
                incoming = []
                for j in candidates:
                    i = int(rows[j])
                    incoming.append(dict(**source_meta, source_line=int(lines[i]), label=label,
                        score=float(scores[i]), priority=float(keys[j]),
                        columns=X.indices[X.indptr[i]:X.indptr[i+1]].tolist()))
                self.state[name][label] = sorted(previous+incoming, key=lambda r:r['priority'])[:limit]
        if sum(len(r['columns']) for name in ('probe','extremes') for pool in self.state[name] for r in pool) > self.config['max_probe_nnz']:
            raise MemoryError('diagnostic sparse budget exceeded; reduce probe/example limits in a new run directory')
        if threshold is not None:
            predicted = scores >= threshold
            counts = [int(np.sum(predicted & (y==1))), int(np.sum(predicted & (y==0))),
                      int(np.sum(~predicted & (y==1))), int(np.sum(~predicted & (y==0)))]
            for kind, key in [('day',source_meta['day']),('host',source_meta['host'])]:
                group = self.state['groups'].setdefault(kind, {}).setdefault(key, [0,0,0,0])
                self.state['groups'][kind][key] = [a+b for a,b in zip(group, counts)]
        self.state['rng'] = self.rng.bit_generator.state

    def matrix(self):
        records = self.state['probe'][0]+self.state['probe'][1]
        ptr = np.r_[0, np.cumsum([len(r['columns']) for r in records])]
        columns = np.array([c for r in records for c in r['columns']],dtype=np.int32)
        X = sparse.csr_matrix((np.ones(len(columns),dtype=np.float32),columns,ptr),
                              shape=(len(records),self.data.width))
        return X, np.array([r['label'] for r in records],dtype=np.uint8)

    def publish(self, directory, split, threshold):
        directory = output_path(directory, self.data.protected)
        ext_vocab_path = self.data.manifest_path.parent / 'assets/vocab_extended.json'
        vocab_path = ext_vocab_path if ext_vocab_path.exists() else self.data.manifest_path.parent / 'assets/vocab.json'
        reverse = {int(v): k for k, v in json.loads(vocab_path.read_text()).items()}
        errors = []
        for label, pool in enumerate(self.state['extremes']):
            for row in pool:
                if (row['score'] >= threshold) == bool(label):
                    continue
                record = {k: v for k, v in row.items() if k != 'priority'}
                record.update(error='FN' if label else 'FP', threshold=threshold,
                              tokens=[reverse.get(c, f"token_{c}") for c in row['columns']])
                errors.append(record)
        atomic_json(directory/f'{split}_errors.json', dict(
            split=split, selection='lowest-score positives / highest-score negatives; NOT a representative error sample',
            maximum_per_error_type=self.config['examples'], examples=errors), self.data.protected)
        atomic_json(directory/f'{split}_diagnostics.json', dict(
            split=split, threshold=threshold, rows_by_label=self.state['rows'],
            empty_rows_by_label=self.state['empty_rows'], groups=self.state['groups'],
            group_count_order=['tp','fp','fn','tn'],
            group_scope='full split at frozen threshold' if split=='test' else 'not collected: validation threshold is selected after the pass',
            probe_counts=[len(x) for x in self.state['probe']], config=self.config), self.data.protected)


def posthoc_learning_curve(data, sample_dir, sample, model, validation_probe,
                           directory, threshold, model_name=None, threads=1, every=25,
                           base_booster=None, base_alpha=None):
    """Evaluate prefixes of the final ensemble, not retraining or early stopping.

    Supports XGBoost, LightGBM, and Random Forest.
    TRAIN probe is drawn from the actual fitted sample. Validation probe is drawn
    during full evaluation. Both use class weights restoring original split priors.
    Negative-tail estimates on small probes are noisy; full metrics remain primary.
    """
    base_alpha = BASE_MARGIN_ALPHA if base_alpha is None else base_alpha
    from sklearn.metrics import average_precision_score, log_loss, roc_auc_score
    directory = output_path(directory, data.protected)
    if every < 1:
        raise ValueError('curve interval must be positive')
    if model_name is None:
        if hasattr(model, 'num_boosted_rounds'):
            model_name = 'xgb'
        elif hasattr(model, 'num_trees'):
            model_name = 'lgbm'
        elif hasattr(model, 'estimators_'):
            model_name = 'rf'
        else:
            raise ValueError(f'unsupported model type: {type(model)}')

    train_probe = XGBDiagnostics(data, **validation_probe.config)
    train_path = directory/'train-probe.json'
    identity = dict(sample_id=sample['sample_id'], runtime=RUNTIME_SHA256,
                    config=train_probe.config, input=data.identity())
    if train_path.exists():
        saved = json.loads(train_path.read_text())
        if saved['identity'] != identity:
            raise ValueError('training probe identity mismatch; use a new diagnostics directory')
        train_probe.restore(saved['state'])
    else:
        with phase('fixed TRAIN diagnostic probe (sample parts only)'):
            for index,part in enumerate(sample['parts']):
                path = Path(sample_dir)/part['path']
                if sha256(path) != part['sha256']:
                    raise ValueError('sample checksum changed during diagnostics')
                with np.load(path,allow_pickle=False) as c:
                    X = sparse.csr_matrix((np.ones(len(c['indices']),dtype=np.float32),c['indices'],c['indptr']),
                                          shape=(len(c['label']), data.width))
                    # Train probe provenance is not exported as raw-source lines.
                    train_probe.update(data.meta['shards'][part['index']],X,c['label'],
                                       c['source_rows'],np.zeros(len(c['label'])))
                if (index+1)%256 == 0:
                    print(f'TRAIN probe: {index+1}/{len(sample["parts"])} parts',flush=True)
        atomic_json(train_path,dict(identity=identity,state=train_probe.state,
                    line_identity='source_rows are zero-based source-shard offsets, not source file lines'),data.protected)

    if model_name == 'xgb':
        import xgboost as xgb
        rounds = model.num_boosted_rounds()
    elif model_name == 'lgbm':
        rounds = model.num_trees()
    elif model_name == 'rf':
        rounds = len(model.estimators_)
    else:
        raise ValueError(f'unrecognized model_name: {model_name}')

    steps = sorted(set([1, rounds, *range(every, rounds+1, every)]))
    curves = []
    for split,probe in [('train',train_probe),('val',validation_probe)]:
        X,y = probe.matrix()
        if len(np.unique(y)) != 2:
            raise ValueError(f'{split} probe needs both classes')
        totals = data.meta['splits'][split]
        weights = np.where(y==1,totals['label_1']/np.sum(y==1),totals['label_0']/np.sum(y==0))
        
        if model_name == 'xgb':
            matrix = xgb.DMatrix(X, nthread=threads)
            preds_by_step = {trees: model.predict(matrix, iteration_range=(0, trees)).astype(np.float64) for trees in steps}
        elif model_name == 'lgbm':
            if base_booster is not None:
                base_n_feats = base_booster.num_feature()
                z_base = base_alpha * base_booster.predict(X[:, :base_n_feats], raw_score=True, num_threads=threads, predict_disable_shape_check=True)
                preds_by_step = {trees: 1.0 / (1.0 + np.exp(-(z_base + model.predict(X, num_iteration=trees, raw_score=True, num_threads=threads)))) for trees in steps}
            else:
                preds_by_step = {trees: model.predict(X, num_iteration=trees, num_threads=threads).astype(np.float64) for trees in steps}
        elif model_name == 'rf':
            step_set = set(steps)
            preds_by_step = {}
            accum = np.zeros(len(y), dtype=np.float64)
            for i, tree in enumerate(model.estimators_, 1):
                accum += tree.predict_proba(X)[:, 1]
                if i in step_set:
                    preds_by_step[i] = (accum / i).astype(np.float64)

        for trees in steps:
            p = preds_by_step[trees]
            pred = p >= threshold
            tp = float(weights[pred & (y==1)].sum()); fp = float(weights[pred & (y==0)].sum())
            fn = float(weights[~pred & (y==1)].sum())
            curves.append(dict(split=split,trees=trees,probe_rows=len(y),
                weighted_logloss=float(log_loss(y,p,sample_weight=weights,labels=[0,1])),
                weighted_average_precision=float(average_precision_score(y,p,sample_weight=weights)),
                weighted_roc_auc=float(roc_auc_score(y,p,sample_weight=weights)),
                weighted_f1_at_final_validation_threshold=2*tp/max(2*tp+fp+fn,1),
                threshold=threshold))
            print(f'Curve {split} trees={trees}: logloss={curves[-1]["weighted_logloss"]:.6g}, AP={curves[-1]["weighted_average_precision"]:.6g}',flush=True)

    if model_name == 'xgb':
        model_sha = hashlib.sha256(model.save_raw()).hexdigest()
        model_cfg = json.loads(model.save_config())
    elif model_name == 'lgbm':
        model_sha = hashlib.sha256(model.model_to_string().encode()).hexdigest()
        model_cfg = model.dump_model()
    else:
        import pickle
        model_sha = hashlib.sha256(pickle.dumps(model)).hexdigest()
        model_cfg = {k: str(v) for k, v in model.get_params().items()}

    report = dict(method='posthoc ensemble-prefix curve on fixed class-stratified uniform probes',
        model_name=model_name,
        identity=identity, model_sha256=model_sha,
        model_config=model_cfg, validation_probe_config=validation_probe.config,
        caveats=['Not full-split metrics; rare false positives may be absent from the probe.',
                 'TRAIN probe comes from fitted undersampled TRAIN; class weighting does not restore missing negative patterns.',
                 'F1 uses the final full-validation threshold at every prefix; logloss/AP are better for studying learning progress.',
                 'No test data, early stopping or automatic hyperparameter decisions are used.'], curve=curves)
    atomic_json(directory/'learning_curve.json',report,data.protected)
    import csv
    def write_csv(path):
        with path.open('w',newline='') as f:
            writer=csv.DictWriter(f,fieldnames=list(curves[0]));writer.writeheader();writer.writerows(curves)
    atomic_write(directory/'learning_curve.csv',write_csv,data.protected)
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        print('matplotlib unavailable; learning_curve.json and .csv are ready',flush=True)
    else:
        fig,axes=plt.subplots(1,2,figsize=(11,4))
        for ax,key in zip(axes,['weighted_logloss','weighted_average_precision']):
            for split in ('train','val'):
                rows=[r for r in curves if r['split']==split]
                ax.plot([r['trees'] for r in rows],[r[key] for r in rows],label=split)
            ax.set(xlabel='Trees',ylabel=key);ax.legend();ax.grid(alpha=.3)
        fig.suptitle(f'{model_name.upper()} fixed probe estimates — not full-split performance');fig.tight_layout()
        atomic_write(directory/'learning_curve.png',lambda p:fig.savefig(p,dpi=150),data.protected)
        plt.close(fig)
    # Scores at every grid threshold from the existing full-validation histogram.
    state_path = directory.parent/'validation-state.npz'
    with np.load(state_path,allow_pickle=False) as saved:
        pos,neg=saved['positive'],saved['negative']
    tp=pos[::-1].cumsum()[::-1];fp=neg[::-1].cumsum()[::-1]
    atomic_write(directory/'validation_threshold_curve.npz',lambda p:np.savez_compressed(p,
        threshold=np.arange(len(pos))/(len(pos)-1),tp=tp,fp=fp,fn=pos.sum()-tp,
        precision=tp/np.maximum(tp+fp,1),recall=tp/max(int(pos.sum()),1),
        f1=2*tp/np.maximum(tp+fp+pos.sum(),1)),data.protected)
    return report


xgb_posthoc_learning_curve = posthoc_learning_curve
ModelDiagnostics = XGBDiagnostics

# END EMBEDDED RUNTIME
meta = json.loads(MANIFEST.read_text())
used_volumes = {p['volume'] for p in meta['shards']}
packaged = all((MANIFEST.parent / 'volumes' / key).is_dir() for key in used_volumes)
if packaged:
    VOLUME_ROOTS = {key: MANIFEST.parent / 'volumes' / key for key in meta['volumes']}
elif (MANIFEST.parent / 'shards').is_dir():
    VOLUME_ROOTS = {key: MANIFEST.parent for key in meta['volumes']}
else:
    VOLUME_ROOTS = None
with phase('open manifest'):
    data = TrainingDataset(MANIFEST, VOLUME_ROOTS)
OUTPUT_ROOT = output_path(OUTPUT_ROOT, data.protected)
SAMPLE_DIR = output_path(SAMPLE_DIR, data.protected)
OUTPUT = output_path(OUTPUT_ROOT / (MODEL_NAME + '-' + PROFILE), [*data.protected, SAMPLE_DIR])
CPU_THREADS = cpu_budget(CPU_THREADS)
LOADER_WORKERS = min(4, max(1, CPU_THREADS // 4))
PREDICT_THREADS = max(1, CPU_THREADS - LOADER_WORKERS)
_blas_limiter = configure_threads(CPU_THREADS)
if DEVICE == 'auto':
    try:
        DEVICE = 'cuda' if subprocess.run(['nvidia-smi'], stdout=subprocess.DEVNULL,
                                          stderr=subprocess.DEVNULL).returncode == 0 else 'cpu'
    except OSError:
        DEVICE = 'cpu'
print(f'Manifest: {MANIFEST}; CPU train={CPU_THREADS}, predict={PREDICT_THREADS}, decode={LOADER_WORKERS}; XGB device={DEVICE}', flush=True)
print(f'Available RAM: {available_bytes()/2**30:.2f} GiB; sample={SAMPLE_DIR}; output={OUTPUT}', flush=True)

# LGBM diagnostics: bounded state included in evaluation checkpoints.
DIAGNOSTICS_ENABLED = True
DIAGNOSTIC_NEGATIVE_ROWS = 10_000
DIAGNOSTIC_POSITIVE_ROWS = 1_000
DIAGNOSTIC_ERROR_EXAMPLES = 5_000  # maximum per FP/FN, each split
DIAGNOSTIC_MAX_NNZ = 5_000_000
LEARNING_CURVE_EVERY = 25
DIAGNOSTICS = OUTPUT / 'diagnostics'
def new_diagnostics():
    return ModelDiagnostics(data, probe_negative=DIAGNOSTIC_NEGATIVE_ROWS,
        probe_positive=DIAGNOSTIC_POSITIVE_ROWS, examples=DIAGNOSTIC_ERROR_EXAMPLES,
        max_probe_nnz=DIAGNOSTIC_MAX_NNZ, seed=17, model_name=MODEL_NAME) if DIAGNOSTICS_ENABLED else None
validation_diagnostics = new_diagnostics()
test_diagnostics = new_diagnostics()

# Resolve Pretrained Checkpoint (ATLASv2 Task 1 Incremental Model)
PRETRAINED_MODEL = None
for cand in [
    MANIFEST.parent / 'pretrained_optc_model' / 'model-slot-0.txt',
    Path('/kaggle/input/datasets/ngkimngn/ml-il-theia/pretrained_optc_model/model-slot-0.txt'),
    Path('/kaggle/input/ml-il-theia/pretrained_optc_model/model-slot-0.txt'),
    MANIFEST.parent / 'model-slot-0.txt',
    *(list(Path('/kaggle/input').rglob('model-slot-0.txt')) if Path('/kaggle/input').exists() else []),
    Path.cwd() / 'IL/dataset_2-THEIA/code/ML/pretrained_optc_model/model-slot-0.txt',
    Path.cwd() / 'IL/dataset_2-THEIA/code/ML/preprocess/output/pretrained_optc_model/model-slot-0.txt',
    Path.cwd() / 'IL/dataset_1-ATLASv2/code/ML/output/kaggle_output/12/results/incremental/model/model-slot-1.txt',
]:
    if cand and cand.exists():
        PRETRAINED_MODEL = cand.resolve()
        break
print(f'Pretrained Checkpoint (ATLASv2 Task 1): {PRETRAINED_MODEL}', flush=True)

In [ ]:
with phase('TRAIN sample preparation / resume'):
    sample = prepare_sample(data, SAMPLE_DIR, target_rows=TRAIN_TARGET_ROWS,
                            neg_per_pos=NEG_PER_POS, seed=SEED,
                            workers=LOADER_WORKERS, max_nnz=MAX_SAMPLE_NNZ)

# 1. Fit Scratch Model (huấn luyện từ đầu trên THEIA)
scratch_dir = OUTPUT / 'scratch'
with phase(f'fit {MODEL_NAME} (Scratch Baseline on THEIA)'):
    model_scratch, raw_predict_scratch, receipt_scratch = fit_sample_model(
        data, SAMPLE_DIR, sample, scratch_dir / 'model', MODEL_NAME, CPU_THREADS,
        max_matrix_bytes=MAX_MATRIX_GIB * 2**30, device=DEVICE if MODEL_NAME == 'xgb' else 'cpu',
        balance_classes=BALANCE_CLASSES, rounds=BOOST_ROUNDS, rf_trees=RF_TREES,
        predict_threads=PREDICT_THREADS, init_model_path=None)
    predict_scratch = ExactPredictor(raw_predict_scratch, enabled=DEDUPE_PREDICTION,
                                     budget_bytes=DEDUPE_BUDGET_MIB * 2**20)

# 2. Small predeclared search. Every candidate sees the same TRAIN sample.
# Selection uses VAL AP, then VAL F1; no Test access, routing or Test threshold search.
import shutil
alpha_fixed = 'ALPHA_OVERRIDE' in os.environ or 'OPTC_ALPHA' in os.environ
weight_fixed = 'THEIA_HARD_NEGATIVE_WEIGHT' in os.environ
candidate_alphas = [BASE_MARGIN_ALPHA] if alpha_fixed else [0.4, 0.8]
candidate_weights = [HARD_NEGATIVE_WEIGHT] if weight_fixed else [1.0, 5.0]
incremental_dir = OUTPUT / 'incremental'
model_il = predict_il = receipt_il = None
candidate_results = []
_candidates = []
if PRETRAINED_MODEL and PRETRAINED_MODEL.exists():
    for alpha in candidate_alphas:
        for hard_weight in candidate_weights:
            directory = OUTPUT / 'incremental_candidates' / f'alpha_{alpha:g}_hardneg_{hard_weight:g}'
            with phase(f'fit LightGBM alpha={alpha}, hard-negative weight={hard_weight}'):
                model, predictor, receipt = fit_sample_model(
                    data, SAMPLE_DIR, sample, directory / 'model', MODEL_NAME, CPU_THREADS,
                    max_matrix_bytes=MAX_MATRIX_GIB * 2**30, device='cpu',
                    balance_classes=BALANCE_CLASSES, rounds=BOOST_ROUNDS,
                    predict_threads=PREDICT_THREADS, init_model_path=PRETRAINED_MODEL,
                    base_margin_alpha=alpha, hard_negative_weight=hard_weight)
            predictor = ExactPredictor(predictor, enabled=DEDUPE_PREDICTION,
                                       budget_bytes=DEDUPE_BUDGET_MIB * 2**20)
            validation = evaluate_resumable(data, 'val', predictor, directory / 'validation-state.npz',
                {'receipt': receipt, 'alpha': alpha, 'hard_negative_weight': hard_weight},
                bins=RANKING_BINS, workers=LOADER_WORKERS)
            candidate_results.append(dict(alpha=alpha, hard_negative_weight=hard_weight,
                                          validation=validation, directory=str(directory)))
            _candidates.append((model, predictor, receipt, directory))
    best = max(range(len(candidate_results)), key=lambda i: (
        candidate_results[i]['validation']['pr_auc_binned'],
        candidate_results[i]['validation']['f1'],
        -candidate_results[i]['hard_negative_weight'], candidate_results[i]['alpha']))
    model_il, raw_predict_il, receipt_il, selected_directory = _candidates[best]
    BASE_MARGIN_ALPHA = candidate_results[best]['alpha']
    HARD_NEGATIVE_WEIGHT = candidate_results[best]['hard_negative_weight']
    shutil.copytree(selected_directory / 'model', incremental_dir / 'model', dirs_exist_ok=True)
    atomic_json(OUTPUT / 'validation_model_selection.json',
                dict(policy='max validation AP, then validation F1; ties prefer weight=1 then alpha=0.8',
                     selected=best, candidates=candidate_results, test_used=False), data.protected)

    # 3. One booster for deployment, algebraically identical to the selected composite.
    sys.path.insert(0, str(MANIFEST.parent / 'source/ML/preprocess'))
    from single_booster import combine_boosters, assert_prediction_parity
    frozen = FrozenAtlasModel(PRETRAINED_MODEL)
    components = [(frozen.anchor, BASE_MARGIN_ALPHA * frozen.alpha),
                  (frozen.residual, BASE_MARGIN_ALPHA), (model_il, 1.0)]
    deployed_model = combine_boosters(components)
    parity_error = 0.0
    # Structural export checks on TRAIN/VAL and old-domain input probes.
    # These checks compare predictions, never select hyperparameters or thresholds.
    for manifest_path in (MANIFEST, OPTC_TEST_MANIFEST, ATLASV2_TEST_MANIFEST):
        probe_data = TrainingDataset(manifest_path, volume_roots={'local': manifest_path.parent})
        for part in probe_data.meta['shards']:
            if manifest_path == MANIFEST and part['split'] == 'test':
                continue
            X_probe, _, _ = decode_shard(probe_data.path_for(part), probe_data.width, part)
            X_probe = FrozenAtlasModel.align(X_probe[:128], data.width)
            parity_error = max(parity_error, assert_prediction_parity(deployed_model, components, X_probe, PREDICT_THREADS))
    deployment_dir = OUTPUT / 'deployment'
    deployment_dir.mkdir(parents=True, exist_ok=True)
    atomic_write(deployment_dir / 'lightgbm.txt', lambda path: deployed_model.save_model(str(path)), data.protected)
    atomic_json(deployment_dir / 'contract.json',
                dict(model='lightgbm.txt', model_sha256=sha256(deployment_dir / 'lightgbm.txt'),
                     model_count=1, feature_width=data.width, vocabulary_sha256=data.meta['vocabulary_sha256'],
                     masked_columns=MASKED_COLUMNS, task2_alpha=BASE_MARGIN_ALPHA,
                     hard_negative_weight=HARD_NEGATIVE_WEIGHT, maximum_probe_margin_error=parity_error,
                     socket_policy='context bit only; no IP/port/protocol features',
                     context_policy=json.loads((MANIFEST.parent / 'assets/vocab_metadata.json').read_text())['host_context_policy']), data.protected)
    shutil.copy2(MANIFEST.parent / 'assets/vocab_extended.json', deployment_dir / 'vocab_extended.json')
    del _candidates
    predict_il = ExactPredictor(lambda X: deployed_model.predict(X, num_threads=PREDICT_THREADS),
                               enabled=DEDUPE_PREDICTION, budget_bytes=DEDUPE_BUDGET_MIB * 2**20)
else:
    raise RuntimeError('Frozen Task-1 composite is required')


In [ ]:
# Packaged audit helpers operate on provenance sidecars, never features.
sys.path.insert(0, str(MANIFEST.parent / 'source/ML/preprocess'))
from cohort_metrics import grouped_metrics, collision_metrics
atomic_json(OUTPUT / 'train_vector_collisions.json',
            collision_metrics(MANIFEST.parent, [p for p in data.meta['shards'] if p['split'] == 'train'],
                              data.width, decode_shard), data.protected)

# Full evaluation of both models and ATLASv2 Zero-shot comparison
def evaluate_model_pipeline(tag, name, model, predict_fn, receipt, model_dir, optc_init=False, base_booster=None, base_alpha=None):
    base_alpha = BASE_MARGIN_ALPHA if base_alpha is None else base_alpha
    model_identity = {'model_sha256': receipt.get('sha256', 'unknown') if receipt else 'zeroshot',
                      'sample_id': sample['sample_id'], 'task2_alpha': BASE_MARGIN_ALPHA,
                      'use_replay': USE_REPLAY, 'hard_negative_weight': HARD_NEGATIVE_WEIGHT}
    diag_dir = model_dir / 'diagnostics'
    val_diag = ModelDiagnostics(data, probe_negative=DIAGNOSTIC_NEGATIVE_ROWS,
                                probe_positive=DIAGNOSTIC_POSITIVE_ROWS, examples=DIAGNOSTIC_ERROR_EXAMPLES,
                                max_probe_nnz=DIAGNOSTIC_MAX_NNZ, seed=17, model_name=MODEL_NAME) if DIAGNOSTICS_ENABLED else None
    test_diag = ModelDiagnostics(data, probe_negative=DIAGNOSTIC_NEGATIVE_ROWS,
                                 probe_positive=DIAGNOSTIC_POSITIVE_ROWS, examples=DIAGNOSTIC_ERROR_EXAMPLES,
                                 max_probe_nnz=DIAGNOSTIC_MAX_NNZ, seed=17, model_name=MODEL_NAME) if DIAGNOSTICS_ENABLED else None
    
    with phase(f'[{name}] full validation evaluation'):
        validation = evaluate_resumable(data, 'val', predict_fn, model_dir / 'validation-state.npz',
                                        model_identity, bins=RANKING_BINS, workers=LOADER_WORKERS, diagnostics=val_diag)
    atomic_json(model_dir / 'validation_metrics.json', validation, data.protected)
    
    if DIAGNOSTICS_ENABLED and val_diag:
        val_diag.publish(diag_dir, 'val', validation['threshold'])
        if model is not None and not optc_init:
            try:
                posthoc_learning_curve(data, SAMPLE_DIR, sample, model,
                    val_diag, diag_dir, validation['threshold'],
                    model_name=MODEL_NAME, threads=PREDICT_THREADS, every=LEARNING_CURVE_EVERY,
                    base_booster=base_booster, base_alpha=base_alpha)
            except Exception as e:
                print(f'[{name}] posthoc curve skipped: {e}')

    with phase(f'[{name}] full test evaluation'):
        test = evaluate_resumable(data, 'test', predict_fn, model_dir / 'test-state.npz', model_identity,
                                  threshold=validation['threshold'], bins=RANKING_BINS,
                                  workers=LOADER_WORKERS, diagnostics=test_diag)
    
    if DIAGNOSTICS_ENABLED and test_diag:
        test_diag.publish(diag_dir, 'test', validation['threshold'])
        print(f'[{name}] FP/FN traceback examples saved to: {diag_dir}', flush=True)

    report = dict(tag=tag, name=name, model=MODEL_NAME, profile=PROFILE, seed=SEED,
                  split_sha256=data.meta['split_sha256'], vocabulary_sha256=data.meta['vocabulary_sha256'],
                  sample_id=sample['sample_id'],
                  model_checkpoint=receipt,
                  validation=validation, test=test)
    report['grouped_test'] = grouped_metrics(
        MANIFEST.parent, [p for p in data.meta['shards'] if p['split'] == 'test'],
        data.width, decode_shard, predict_fn, validation['threshold'])
    report['experiment'] = dict(task2_alpha=BASE_MARGIN_ALPHA if tag == 'incremental' else None,
                                use_replay=bool(tag == 'incremental' and USE_REPLAY),
                                hard_negative_weight=HARD_NEGATIVE_WEIGHT if tag == 'incremental' else None,
                                balance_classes=BALANCE_CLASSES if tag != 'zeroshot' else None,
                                masked_columns=MASKED_COLUMNS)
    atomic_json(model_dir / f'{MODEL_NAME}_metrics.json', report, data.protected)
    return report

results = {}

# 1. Evaluate Scratch
results['scratch'] = evaluate_model_pipeline(
    'scratch', 'THEIA Scratch (Baseline)', model_scratch, predict_scratch, receipt_scratch, scratch_dir)

# 2. Evaluate Incremental
if predict_il is not None:
    base_booster_il = None
    if PRETRAINED_MODEL and PRETRAINED_MODEL.exists():
        import lightgbm as lgb
        base_booster_il = FrozenAtlasModel(PRETRAINED_MODEL)
    results['incremental'] = evaluate_model_pipeline(
        'incremental', 'ATLASv2 -> THEIA Continual Boosting (IL)', model_il, predict_il, receipt_il, incremental_dir,
        base_booster=base_booster_il, base_alpha=BASE_MARGIN_ALPHA)

# 3. Evaluate ATLASv2 Zero-shot (Trước IL trên THEIA)
if PRETRAINED_MODEL and PRETRAINED_MODEL.exists():
    zeroshot_dir = OUTPUT / 'atlasv2_zeroshot'
    import lightgbm as lgb
    atlas_booster = FrozenAtlasModel(PRETRAINED_MODEL)
    atlas_feat_count = atlas_booster.num_feature()  # 12,337
    atlas_predict = ExactPredictor(lambda X: atlas_booster.predict(X[:, :atlas_feat_count], num_threads=PREDICT_THREADS),
                                  enabled=DEDUPE_PREDICTION, budget_bytes=DEDUPE_BUDGET_MIB * 2**20)
    results['zeroshot'] = evaluate_model_pipeline(
        'zeroshot', 'ATLASv2 Base Model (Zero-shot / Trước IL trên THEIA)', atlas_booster, atlas_predict,
        {'sha256': sha256(PRETRAINED_MODEL), 'path': PRETRAINED_MODEL.name}, zeroshot_dir, optc_init=True)

# Publish the validation-selected threshold beside the one deployment model.
contract_path = OUTPUT / 'deployment/contract.json'
contract = json.loads(contract_path.read_text())
contract.update(threshold=results['incremental']['validation']['threshold'],
                threshold_source='validation_f1_max_constrained',
                old_domain_fixed_thresholds={'optc': 0.87427, 'atlasv2': 0.5})
atomic_json(contract_path, contract, data.protected)

# Build Executive Comparison Summary
import csv
summary_rows = []
for tag, rep in results.items():
    t = rep['test']
    v = rep['validation']
    conf = t['confusion']
    tn, fp, fn, tp = conf['tn'], conf['fp'], conf['fn'], conf['tp']
    fpr = fp / max(1, fp + tn)
    val_ap = v.get('pr_auc_binned', v.get('weighted_average_precision', 0.0))
    val_f1 = v.get('f1', v.get('weighted_f1', 0.0))
    test_ap = t.get('pr_auc_binned', t.get('weighted_average_precision', 0.0))
    test_roc = t.get('roc_auc_binned', t.get('weighted_roc_auc', 0.0))
    test_f1 = t.get('f1', t.get('weighted_f1', 0.0))
    test_prec = t.get('precision', t.get('weighted_precision', 0.0))
    test_rec = t.get('recall', t.get('weighted_recall', 0.0))
    test_fpr = t.get('fpr', fpr)

    summary_rows.append({
        'Model Experiment': rep['name'],
        'Val AP': round(float(val_ap), 4),
        'Val F1': round(float(val_f1), 4),
        'Threshold': round(float(v.get('threshold', 0.5)), 4),
        'Test AP': round(float(test_ap), 4),
        'Test ROC-AUC': round(float(test_roc), 4),
        'Test F1': round(float(test_f1), 4),
        'Test Precision': round(float(test_prec), 4),
        'Test Recall': round(float(test_rec), 4),
        'Test FPR': f'{test_fpr:.4%}',
        'TP': int(tp), 'FP': int(fp), 'FN': int(fn), 'TN': int(tn),
    })

print('\n' + '=' * 105)
print('EXECUTIVE COMPARISON REPORT: TRƯỚC IL (ATLASV2 ZERO-SHOT) vs SCRATCH vs SAU IL (THEIA CONTINUAL BOOSTING)')
print('=' * 105)
col_widths = {k: max(len(k), max(len(str(r[k])) for r in summary_rows)) for k in summary_rows[0]}
header_str = ' | '.join(k.ljust(col_widths[k]) for k in summary_rows[0])
print(header_str)
print('-' * len(header_str))
for r in summary_rows:
    print(' | '.join(str(r[k]).ljust(col_widths[k]) for k in r))
print('=' * 105)

with (OUTPUT / 'comparison_metrics.csv').open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=list(summary_rows[0].keys()))
    writer.writeheader()
    writer.writerows(summary_rows)

# ------------------------------------------------------------------------------
# 3b. Secondary predeclared operating point: calibrate on VAL only.
# Never search test histograms for a threshold.
soc_rows = []
for tag, name, directory in [('zeroshot', 'ATLASv2 Zero-shot', OUTPUT / 'atlasv2_zeroshot'),
                              ('scratch', 'THEIA Scratch', OUTPUT / 'scratch'),
                              ('incremental', 'THEIA IL', OUTPUT / 'incremental')]:
    val_path, test_path = directory / 'validation-state.npz', directory / 'test-state.npz'
    if not val_path.exists() or not test_path.exists():
        continue
    with np.load(val_path, allow_pickle=False) as state:
        val_positive, val_negative = state['positive'], state['negative']
    with np.load(test_path, allow_pickle=False) as state:
        positive, negative = state['positive'], state['negative']
    for budget in (0.005, 0.01):
        threshold, index = select_fpr_threshold(val_negative, budget)
        tp, fp = int(positive[index:].sum()), int(negative[index:].sum())
        fn, tn = int(positive.sum()) - tp, int(negative.sum()) - fp
        soc_rows.append(dict(model=name, threshold=threshold, calibration_split='val',
                             target_validation_fpr=budget, test_f1=2*tp/max(2*tp+fp+fn,1),
                             test_fpr=fp/max(fp+tn,1), tp=tp, fp=fp, fn=fn, tn=tn))
    val_tps, val_fps = val_positive[::-1].cumsum()[::-1], val_negative[::-1].cumsum()[::-1]
    val_total_p = val_positive.sum()
    if val_total_p > 0:
        val_f1s = 2 * val_tps / np.maximum(2 * val_tps + val_fps + (val_total_p - val_tps), 1)
        best_f1_idx = int(np.argmax(val_f1s))
        f1_th = best_f1_idx / (len(val_positive) - 1)
        tp, fp = int(positive[best_f1_idx:].sum()), int(negative[best_f1_idx:].sum())
        fn, tn = int(positive.sum()) - tp, int(negative.sum()) - fp
        soc_rows.append(dict(model=name, threshold=f1_th, calibration_split='val',
                             target_validation_fpr='f1_max', test_f1=2*tp/max(2*tp+fp+fn,1),
                             test_fpr=fp/max(fp+tn,1), tp=tp, fp=fp, fn=fn, tn=tn))
atomic_json(OUTPUT / 'validation_calibrated_operating_points.json', soc_rows, data.protected)
print('Validation-calibrated operating points:', soc_rows)

# ------------------------------------------------------------------------------
# 4. Backward Transfer (BWT) Evaluation on OpTC and ATLASv2 Test Sets (Retention)
# ------------------------------------------------------------------------------
bwt_results = {}
if PRETRAINED_MODEL and PRETRAINED_MODEL.exists():
    import lightgbm as lgb
    from scipy import sparse
    base_booster = FrozenAtlasModel(PRETRAINED_MODEL)
    base_feat_count = base_booster.num_feature()

    # 4a. BWT on OpTC Test Set (Task 0 Retention)
    if OPTC_TEST_MANIFEST and OPTC_TEST_MANIFEST.exists():
        with phase('Backward Transfer Evaluation on OpTC Test Set'):
            optc_test_data = TrainingDataset(OPTC_TEST_MANIFEST, volume_roots={'local': OPTC_TEST_MANIFEST.parent})
            print(f"OpTC Test Data: {optc_test_data.meta['splits']['test']['rows']:,} rows, {optc_test_data.meta['splits']['test']['label_1']:,} positives", flush=True)

            optc_bwt_dir = OUTPUT / 'bwt_optc_test'
            optc_bwt_dir.mkdir(parents=True, exist_ok=True)
            OPTC_FROZEN_TH = 0.87427

            optc_base_pred_fn = ExactPredictor(
                lambda X: base_booster.predict(X[:, :base_feat_count] if X.shape[1] >= base_feat_count else X, num_threads=PREDICT_THREADS, predict_disable_shape_check=True),
                enabled=DEDUPE_PREDICTION, budget_bytes=DEDUPE_BUDGET_MIB * 2**20
            )
            print("Evaluating Pretrained Model (Trước IL) on OpTC Test...", flush=True)
            res_optc_before = evaluate_resumable(
                optc_test_data, 'test', optc_base_pred_fn, optc_bwt_dir / 'optc_before_state.npz',
                {'composition_sha256': sha256(PRETRAINED_MODEL.parent / 'composition.json'), 'model_sha256': sha256(PRETRAINED_MODEL), 'split': 'test'}, threshold=OPTC_FROZEN_TH, bins=RANKING_BINS, workers=LOADER_WORKERS
            )
            bwt_results['optc_before_il'] = res_optc_before

            if predict_il is not None and model_il is not None:
                def predict_il_optc(X_optc):
                    if X_optc.shape[1] < data.width:
                        X_full = sparse.csr_matrix((X_optc.data, X_optc.indices, X_optc.indptr), shape=(X_optc.shape[0], data.width))
                    else:
                        X_full = X_optc
                    return deployed_model.predict(X_full, num_threads=PREDICT_THREADS)

                il_optc_pred_fn = ExactPredictor(
                    predict_il_optc, enabled=DEDUPE_PREDICTION, budget_bytes=DEDUPE_BUDGET_MIB * 2**20
                )
                print(f"Evaluating Continual Model (Sau IL) on OpTC Test at threshold {OPTC_FROZEN_TH:.4f}...", flush=True)
                res_optc_after = evaluate_resumable(
                    optc_test_data, 'test', il_optc_pred_fn, optc_bwt_dir / 'optc_after_state.npz',
                    {'composition_sha256': sha256(PRETRAINED_MODEL.parent / 'composition.json'), 'base_sha256': sha256(PRETRAINED_MODEL), 'residual': receipt_il, 'task2_alpha': BASE_MARGIN_ALPHA, 'split': 'test'}, threshold=OPTC_FROZEN_TH, bins=RANKING_BINS, workers=LOADER_WORKERS
                )
                bwt_results['optc_after_il'] = res_optc_after
                bwt_f1_optc = res_optc_after['f1'] - res_optc_before['f1']
                bwt_ap_optc = res_optc_after['pr_auc_binned'] - res_optc_before['pr_auc_binned']
                print("\n" + "=" * 90)
                print("BACKWARD TRANSFER (BWT) / RETENTION REPORT ON OPTC TEST SET (TASK 0)")
                print("=" * 90)
                print(f"  Trước IL (ATLASv2 Base) : F1 = {res_optc_before['f1']:.4%}, AP = {res_optc_before['pr_auc_binned']:.4%}")
                print(f"  Sau IL (THEIA Model)    : F1 = {res_optc_after['f1']:.4%}, AP = {res_optc_after['pr_auc_binned']:.4%}")
                print(f"  Backward Transfer (BWT F1): {bwt_f1_optc:+.4%}")
                print(f"  Backward Transfer (BWT AP): {bwt_ap_optc:+.4%}")
                print("=" * 90)

    # 4b. BWT on ATLASv2 Test Set (Task 1 Retention)
    if ATLASV2_TEST_MANIFEST and ATLASV2_TEST_MANIFEST.exists():
        with phase('Backward Transfer Evaluation on ATLASv2 Test Set'):
            atlas_test_data = TrainingDataset(ATLASV2_TEST_MANIFEST, volume_roots={'local': ATLASV2_TEST_MANIFEST.parent})
            print(f"ATLASv2 Test Data: {atlas_test_data.meta['splits']['test']['rows']:,} rows, {atlas_test_data.meta['splits']['test']['label_1']:,} positives", flush=True)

            atlas_bwt_dir = OUTPUT / 'bwt_atlasv2_test'
            atlas_bwt_dir.mkdir(parents=True, exist_ok=True)
            ATLAS_TH = 0.50

            atlas_base_pred_fn = ExactPredictor(
                lambda X: base_booster.predict(X[:, :base_feat_count] if X.shape[1] >= base_feat_count else X, num_threads=PREDICT_THREADS),
                enabled=DEDUPE_PREDICTION, budget_bytes=DEDUPE_BUDGET_MIB * 2**20
            )
            print("Evaluating Pretrained Model (Trước IL) on ATLASv2 Test...", flush=True)
            res_atlas_before = evaluate_resumable(
                atlas_test_data, 'test', atlas_base_pred_fn, atlas_bwt_dir / 'atlas_before_state.npz',
                {'composition_sha256': sha256(PRETRAINED_MODEL.parent / 'composition.json'), 'model_sha256': sha256(PRETRAINED_MODEL), 'split': 'test'}, threshold=ATLAS_TH, bins=RANKING_BINS, workers=LOADER_WORKERS
            )
            bwt_results['atlas_before_il'] = res_atlas_before

            if predict_il is not None and model_il is not None:
                def predict_il_atlas(X_atlas):
                    if X_atlas.shape[1] < data.width:
                        X_full = sparse.csr_matrix((X_atlas.data, X_atlas.indices, X_atlas.indptr), shape=(X_atlas.shape[0], data.width))
                    else:
                        X_full = X_atlas
                    return deployed_model.predict(X_full, num_threads=PREDICT_THREADS)

                il_atlas_pred_fn = ExactPredictor(
                    predict_il_atlas, enabled=DEDUPE_PREDICTION, budget_bytes=DEDUPE_BUDGET_MIB * 2**20
                )
                print(f"Evaluating Continual Model (Sau IL) on ATLASv2 Test at threshold {ATLAS_TH:.4f}...", flush=True)
                res_atlas_after = evaluate_resumable(
                    atlas_test_data, 'test', il_atlas_pred_fn, atlas_bwt_dir / 'atlas_after_state.npz',
                    {'composition_sha256': sha256(PRETRAINED_MODEL.parent / 'composition.json'), 'base_sha256': sha256(PRETRAINED_MODEL), 'residual': receipt_il, 'task2_alpha': BASE_MARGIN_ALPHA, 'split': 'test'}, threshold=ATLAS_TH, bins=RANKING_BINS, workers=LOADER_WORKERS
                )
                bwt_results['atlas_after_il'] = res_atlas_after
                bwt_f1_atlas = res_atlas_after['f1'] - res_atlas_before['f1']
                bwt_ap_atlas = res_atlas_after['pr_auc_binned'] - res_atlas_before['pr_auc_binned']
                print("\n" + "=" * 90)
                print("BACKWARD TRANSFER (BWT) / RETENTION REPORT ON ATLASV2 TEST SET (TASK 1)")
                print("=" * 90)
                print(f"  Trước IL (ATLASv2 Base) : F1 = {res_atlas_before['f1']:.4%}, AP = {res_atlas_before['pr_auc_binned']:.4%}")
                print(f"  Sau IL (THEIA Model)    : F1 = {res_atlas_after['f1']:.4%}, AP = {res_atlas_after['pr_auc_binned']:.4%}")
                print(f"  Backward Transfer (BWT F1): {bwt_f1_atlas:+.4%}")
                print(f"  Backward Transfer (BWT AP): {bwt_ap_atlas:+.4%}")
                print("=" * 90)

    atomic_json(OUTPUT / 'bwt_metrics.json', bwt_results, data.protected)
    retention = {}
    for domain in ('optc', 'atlas'):
        before, after = bwt_results.get(domain + '_before_il'), bwt_results.get(domain + '_after_il')
        if before is not None and after is not None:
            delta = {metric: after[metric] - before[metric]
                     for metric in ('f1', 'precision', 'recall', 'pr_auc_binned', 'roc_auc_binned')}
            retention[domain] = dict(delta=delta, no_measured_degradation=all(v >= 0 for v in delta.values()))
    atomic_json(OUTPUT / 'retention_deltas.json', dict(domains=retention,
                use='Final diagnostic only; never choose thresholds/checkpoints using old-domain TEST',
                predictor='composite frozen Task 1 vs Task 2; not a full continual-learning BWT matrix'), data.protected)
    print('Measured retention deltas (no absolute preservation claim):', retention)


In [ ]:
# Learning curves for scratch and incremental models
from IPython.display import display, Image
for tag, mdir in [('Scratch', scratch_dir), ('Incremental', incremental_dir)]:
    img = mdir / 'diagnostics' / 'learning_curve.png'
    if img.exists():
        print(f"=== Learning Curve: {tag} ===")
        display(Image(filename=str(img)))


In [ ]:
# Plot side-by-side Confusion Matrix comparison
import matplotlib.pyplot as plt
import numpy as np

try:
    plot_items = []
    if 'zeroshot' in results:
        plot_items.append(('ATLASv2 Model (Zero-shot)', results['zeroshot']['test']['confusion']))
    if 'scratch' in results:
        plot_items.append(('THEIA Scratch (Baseline)', results['scratch']['test']['confusion']))
    if 'incremental' in results:
        plot_items.append(('ATLASv2 -> THEIA (Sau IL)', results['incremental']['test']['confusion']))
    if 'optc_after_il' in bwt_results:
        plot_items.append(('Sau IL trên OpTC Test (BWT)', bwt_results['optc_after_il']['confusion']))
    if 'atlas_after_il' in bwt_results:
        plot_items.append(('Sau IL trên ATLASv2 Test (BWT)', bwt_results['atlas_after_il']['confusion']))

    if plot_items:
        n_plots = len(plot_items)
        fig, axes = plt.subplots(1, n_plots, figsize=(4.5 * n_plots, 4.0))
        if n_plots == 1:
            axes = [axes]
        for ax, (title, conf) in zip(axes, plot_items):
            matrix = np.array([[conf['tn'], conf['fp']], [conf['fn'], conf['tp']]])
            im = ax.imshow(matrix, cmap='Blues', interpolation='nearest')
            ax.set_title(title, fontsize=10, fontweight='bold')
            ax.set_xticks([0, 1])
            ax.set_yticks([0, 1])
            ax.set_xticklabels(['Benign', 'Attack'], fontsize=9)
            ax.set_yticklabels(['Benign', 'Attack'], fontsize=9)
            for i in range(2):
                for j in range(2):
                    val = matrix[i, j]
                    ax.text(j, i, f'{val:,}', ha='center', va='center',
                            color='white' if val > matrix.max() / 2 else 'black',
                            fontweight='bold', fontsize=10)
        plt.tight_layout()
        plt.show()
except Exception as e:
    print(f'Plotting confusion matrix skipped: {e}')


In [ ]:
# Package model outputs, comparisons, and source-line provenance for offline investigation.
import json, hashlib, os, zipfile
from pathlib import Path

zip_path = output_path(Path(f'{MODEL_NAME}_{PROFILE}_results.zip'), data.protected)
temporary = output_path(zip_path.with_suffix('.zip.tmp'), data.protected)
assets = data.manifest_path.parent / 'assets'
sources = {entry['source_id']: entry for entry in data.meta.get('sources', [])}
shard_index = {}
for part in data.meta['shards']:
    source = sources.get(part['source_id'], {})
    key = f"{part['volume']}/{part['path']}"
    if key in shard_index:
        raise ValueError(f'Duplicate shard identity: {key}')
    shard_index[key] = dict(
        source_id=part['source_id'],
        source_gz=source.get('source', {}).get('path', 'DATASET/DARPA-TC-E3-THEIA/CSV/event_table.csv') if isinstance(source, dict) else 'DATASET/DARPA-TC-E3-THEIA/CSV/event_table.csv',
        host=source.get('file_host', 'theia-desktop') if isinstance(source, dict) else 'theia-desktop',
        day=source.get('day', 'THEIA-mixed-days') if isinstance(source, dict) else 'THEIA-mixed-days',
        shard=part['path'],
        volume=part['volume'],
        split=part['split'],
        label_0=part['label_0'],
        label_1=part['label_1'],
    )

try:
    with zipfile.ZipFile(temporary, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
        for src in sorted(Path(OUTPUT).rglob('*')):
            if src.is_file() and src.resolve() not in {temporary, zip_path}:
                archive.write(src, 'results/' + src.relative_to(OUTPUT).as_posix())
        for src, name in ((data.manifest_path, 'shard_manifest.json'),
                          (assets/'vocab.json', 'vocab.json'),
                          (assets/'vocab_metadata.json', 'vocab_metadata.json')):
            if src.exists():
                archive.write(src, 'provenance/' + name)
        if (assets / 'vocab_extended.json').exists():
            archive.write(assets / 'vocab_extended.json', 'provenance/vocab_extended.json')
        split_asset = assets / 'split_manifest.json'
        if split_asset.exists():
            archive.write(split_asset, 'provenance/split_manifest.json')
        # Residual checkpoints alone cannot reproduce Task 2 inference.
        for src in sorted(PRETRAINED_MODEL.parent.iterdir()):
            if src.is_file():
                archive.write(src, 'frozen_task1/' + src.name)
        archive.writestr('inference_contract.json', json.dumps(dict(
            deployment_model='results/deployment/lightgbm.txt', deployment_model_count=1,
            deployment_contract='results/deployment/contract.json',
            task1_composition='frozen_task1/composition.json', task2_alpha=BASE_MARGIN_ALPHA,
            task2_residual_receipt=receipt_il,
            task2_formula='sigmoid(task2_alpha * frozen_task1_raw_score + z_theia_residual)',
            use_replay=USE_REPLAY, hard_negative_weight=HARD_NEGATIVE_WEIGHT,
            threshold=results.get('incremental', {}).get('validation', {}).get('threshold'),
            threshold_source='validation_f1_max_constrained'), indent=2))
        archive.writestr('provenance/shard_source_index.json', json.dumps(shard_index, indent=2))
        archive.writestr('provenance/README.txt',
            'Diagnostics use source_id and source_line (1-based physical CSV line, including header).\n'
            'Join source_id to shard_manifest.json sources, then stream source.path locally.\n'
            'shard_source_index.json keys are volume/path; labels do not determine source roots.\n')
    os.replace(temporary, zip_path)
finally:
    temporary.unlink(missing_ok=True)
